<a href="https://colab.research.google.com/github/nathanaelbrown/Source_code/blob/master/Project_1_APPLE_HBR_Report.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Configure runtime and install necessary software

APPLE HBR Report - Project 1
    
Before loading and running this notebook set the environment runtime: T4 GPU and Python3

In [ ]:
# =====================================================================
# STEP 1: CONFIGURE RUNTIME AND INSTALL NECESSARY SOFTWARE
# =====================================================================
# This section prepares our environment with the right tools.
# We are installing external packages that Python doesn't come with by default.

import os
import sys

print("Installing llama-cpp-python, LangChain, and machine learning backends...")

# 1. Install llama-cpp-python using a pre-compiled GPU-accelerated wheel.
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122 -q

# 2. Install modern, fully compatible versions of LangChain, sentence-transformers, and scikit-learn
# keeping NumPy 2.x support intact so that langchain-community dependencies are fully satisfied.
!uv pip install langchain==0.3.27 \
            langchain-community==0.3.27 \
            torch==2.6.0 \
            sentence_transformers==3.4.1 \
            scikit-learn>=1.5.0 \
            faiss-cpu==1.12.0 \
            pypdf==6.1.3 \
            huggingface_hub==0.35.3 \
            torchvision==0.21.0 -q

print("\n[System] All packages successfully installed!")
print("[System] Automatically restarting the Colab session now to activate clean imports...")

# Force a clean runtime restart on the fly without erasing locally saved files
os.kill(os.getpid(), 9)

Installing llama-cpp-python, LangChain, and machine learning backends...


### ⚡ Automatic Runtime Setup:
The cell above installs all necessary backend tools and automatically restarts your runtime session behind the scenes to load them. There is no need for manual menu interaction! You can safely proceed with running the document loader and subsequent steps once the installation completes.

In [ ]:
# RetrievalQA allows us to ask questions to our document and retrieve answers
from langchain.chains import RetrievalQA

# This converts human words into lists of numbers (embeddings) that computers can analyze
from langchain.embeddings import SentenceTransformerEmbeddings

# This helps us break large documents into smaller, manageable paragraphs or chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

# This is our local database designed specifically to search through text embeddings quickly
from langchain.vectorstores import FAISS

# We import both loaders so we can choose to load either standard text files or PDF reports
from langchain.document_loaders import TextLoader, PyPDFLoader

# This helps us download pre-trained open-source AI models from the Hugging Face Hub
from huggingface_hub import hf_hub_download

# This allows us to run local Llama models directly on our machine/GPU
from langchain.llms import LlamaCpp

# Pandas helps us build, format, and display clean summary tables and dataframes
import pandas as pd

## Load the Data (Dynamic Upload & Automatic Remote Fetch as Default)

In [ ]:
import os
import requests
from google.colab import files

def get_document():
    default_filename = 'HBR_How_Apple_Is_Organized_For_Innovation.pdf'
    # Publicly accessible copy of the HBR article
    fallback_url = 'https://hbr.org/resources/pdfs/comm/apple/organization-for-innovation.pdf'

    print("=== Document Loader Option ===")
    print("Would you like to:")
    print("1. Use the default 'How Apple Is Organized for Innovation' PDF")
    print("2. Upload your own document (PDF or TXT)")

    choice = input("Enter 1 or 2: ").strip()

    if choice == '2':
        print("\nPlease upload your PDF or TXT file...")
        uploaded = files.upload()
        if not uploaded:
            print("No file uploaded. Falling back to default document.")
            return get_default_file(default_filename, fallback_url)
        else:
            uploaded_filename = list(uploaded.keys())[0]
            print(f"\nSuccessfully uploaded: {uploaded_filename}")
            return uploaded_filename
    else:
        return get_default_file(default_filename, fallback_url)

def get_default_file(filename, url):
    if os.path.exists(filename):
        print(f"\n[Status] Found '{filename}' already present in the local file system.")
        return filename
    else:
        print(f"\n[Status] '{filename}' not found locally.")
        print("Downloading the default document remotely...")
        try:
            response = requests.get(url, timeout=15)
            response.raise_for_status()
            with open(filename, 'wb') as f:
                f.write(response.content)
            print("Download complete!")
            return filename
        except Exception as e:
            print(f"Error downloading from remote source: {e}")
            print("Please upload the file manually.")
            return None

# Run the selection and load the appropriate document
doc_path = get_document()

if doc_path:
    if doc_path.lower().endswith('.pdf'):
        print(f"Loading PDF document using PyPDFLoader...")
        loader = PyPDFLoader(doc_path)
    else:
        print(f"Loading text document using TextLoader...")
        loader = TextLoader(doc_path, encoding='utf-8')

    data = loader.load()
    print(f"Successfully loaded {len(data)} pages/documents!")
else:
    print("No document loaded. Please execute this cell again and choose an option.")

## Split the Extracted Data into Text Chunks

In [ ]:
# =====================================================================
# STEP 3: SPLIT THE DOCUMENT INTO MANAGEABLE CHUNKS
# =====================================================================
# Think of this like cutting a long book chapter into short, bite-sized
# paragraphs. AI models have a limit to how much text they can read at
# once (their memory window). Splitting ensures we only feed the AI the
# most relevant paragraphs to answer our questions.

def get_split_parameters():
    print("=== Text Splitter Settings ===")
    print("You can customize how the document is sliced, or press Enter to keep the defaults.")

    # 1. Ask for chunk size (default: 2000 characters)
    chunk_size_input = input("Enter chunk size (default 2000): ").strip()
    chunk_size = int(chunk_size_input) if chunk_size_input.isdigit() else 2000

    # 2. Ask for overlap (default: 200 characters)
    overlap_input = input("Enter chunk overlap (default 200): ").strip()
    chunk_overlap = int(overlap_input) if overlap_input.isdigit() else 200

    print(f"\n[Config] Using chunk_size = {chunk_size} and chunk_overlap = {chunk_overlap}\n")
    return chunk_size, chunk_overlap

# Retrieve the user's settings or defaults
chunk_size, chunk_overlap = get_split_parameters()

# Initialize the recursive character text splitter.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)

# Perform the actual split on our loaded document data
text_chunks = text_splitter.split_documents(data)
print(f"Successfully split the document into {len(text_chunks)} chunks!\n")

# Generate and display a structured summary table including native page numbers
chunk_data = []
for i, chunk in enumerate(text_chunks):
    # Extract source page metadata (PyPDFLoader starts page index at 0, so we add 1 for human view)
    native_page = chunk.metadata.get('page', 0) + 1 if 'page' in chunk.metadata else "N/A (TXT File)"
    chunk_data.append({
        "Chunk Number": i,
        "Chunk Length (Characters)": len(chunk.page_content),
        "Native Page Number": native_page
    })

df_chunks = pd.DataFrame(chunk_data)

# Format and display the table styled with headers and borders
styled_df = df_chunks.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#4F81BD'), ('color', 'white'), ('font-weight', 'bold'), ('border', '1px solid black')]},
    {'selector': 'td', 'props': [('border', '1px solid black'), ('text-align', 'center')]}
]).hide(axis='index')

display(styled_df)

In [ ]:
# =====================================================================
# STEP 3B: INTERACTIVELY INSPECT INDIVIDUAL CHUNKS
# =====================================================================
# This cell lets you inspect the actual text content of any chunk
# created above. Simply input a chunk number to see what text it contains!

def inspect_chunks():
    total_chunks = len(text_chunks)
    print(f"=== Chunk Inspector (Total Chunks Available: {total_chunks}) ===")

    while True:
        user_input = input(f"Enter a chunk index to inspect (0 to {total_chunks - 1}) or 'exit' to quit: ").strip()
        if user_input.lower() == 'exit':
            break

        if user_input.isdigit():
            idx = int(user_input)
            if 0 <= idx < total_chunks:
                selected_chunk = text_chunks[idx]
                # Retrieve native page number metadata (defaulting to 1 if not present)
                page_num = selected_chunk.metadata.get('page', 0) + 1

                print("\n" + "="*60)
                print(f" CHUNK #{idx} DETAILS  (Source Page: {page_num})")
                print("="*60)
                print(selected_chunk.page_content)
                print("="*60 + "\n")
            else:
                print(f"Invalid index. Please enter a number between 0 and {total_chunks - 1}.")
        else:
            print("Please enter a valid numeric index.")

inspect_chunks()

### Define Queries for Analysis

In [ ]:
query1 = "Who are the authors of this article and who published this article?"
query2 = "List down the three leadership characteristics in bulleted points and explain each one of the characteristics under two lines."
query3 = "Can you explain specific examples from the article where Apple's approach to leadership has led to successful innovations?"

print("Queries defined successfully:")
print(f"Query 1: {query1}")
print(f"Query 2: {query2}")
print(f"Query 3: {query3}")

## Load the embedding model

In [ ]:
# =====================================================================
# STEP 4: LOAD THE EMBEDDING MODEL
# =====================================================================
# This model will translate our text chunks into numerical vectors
# so we can perform fast mathematical similarity searches.

from langchain.embeddings import SentenceTransformerEmbeddings

# Initialize our high-quality open-source text embedding model
# This model converts words into math vectors of 768 dimensions
embeddings = SentenceTransformerEmbeddings(model_name="BAAI/bge-base-en-v1.5")
print("Embedding model 'BAAI/bge-base-en-v1.5' loaded successfully!")

ImportError: Could not import sentence_transformers python package. Please install it with `pip install sentence-transformers`.

## Create Embeddings for each of the Text Chunk

In [ ]:
vector_store = FAISS.from_documents(text_chunks, embedding=embeddings)

In [ ]:
text_chunks[0].page_content

'Item 7. Management’s Discussion and Analysis of Financial Condition and Results of Operations The following discussion should be read in conjunction with the consolidated financial statements and accompanying notes included in Part II, Item 8 of this Form 10-K. This section of this Form 10-K generally discusses 2020 and 2019 items and year-to-year comparisons between 2020 and 2019. Discussions of 2018 items and year-to-year comparisons between 2019 and 2018 that are not included in this Form 10-K can be found in “Management’s Discussion and Analysis of Financial Condition and Results of Operations” in Part II, Item 7 of the Company’s Annual Report on Form 10-K for the fiscal year ended September 28, 2019. Fiscal Year Highlights COVID-19 Update COVID-19 has spread rapidly throughout the world, prompting governments and businesses to take unprecedented measures in response. Such measures have included restrictions on travel and business operations, temporary closures of businesses, and 

In [ ]:
vector_store.index.reconstruct(0)

array([ 4.16787341e-03, -1.48787340e-02, -8.85816757e-03, -1.56945474e-02,
        7.74797499e-02, -3.21615976e-03,  2.07387730e-02,  6.89300522e-02,
       -1.58986300e-02, -1.77491140e-02, -4.94190007e-02, -2.90449671e-02,
       -6.10513352e-02,  4.57197540e-02, -1.75454412e-02,  3.32643427e-02,
        5.34815527e-02,  2.74204910e-02,  2.84301862e-02,  1.22596830e-04,
       -6.59145042e-02, -4.30929996e-02,  5.00659049e-02,  6.72984915e-03,
        5.43527752e-02, -3.26178372e-02,  1.49575407e-02, -1.83257945e-02,
       -4.23273928e-02,  6.82438537e-03,  1.63538719e-03,  2.54531018e-02,
       -9.14115750e-04,  2.26244703e-02,  4.45513763e-02,  8.48012883e-03,
        1.79633200e-02,  1.88731626e-02, -2.90709902e-02, -3.47838737e-02,
       -4.17627990e-02,  1.83200161e-03, -5.58526441e-02, -1.81655604e-02,
       -1.58502646e-02,  2.15283339e-03, -7.14306859e-03,  1.73524953e-02,
       -1.08003784e-02, -3.33475955e-02, -3.47001366e-02, -1.30684525e-02,
       -2.43773367e-02, -

## Load the LLM

In [ ]:
# Define the model repository and the file name from HuggingFace Hub
model_name_or_path = "TheBloke/Llama-2-13B-chat-GGUF"
model_basename = "llama-2-13b-chat.Q5_K_M.gguf"

# Download the model file from HuggingFace Hub
model_path = hf_hub_download(
    repo_id=model_name_or_path,
    filename=model_basename
)

llama-2-13b-chat.Q5_K_M.gguf:   0%|          | 0.00/9.23G [00:00<?, ?B/s]

In [ ]:
# Initialize the LlamaCpp model with configuration
llm = LlamaCpp(
    model_path=model_path,
    temperature=0.01,
    top_p=0.95,
    verbose=False,
    n_ctx=4096,
    n_batch=1024,
    n_gpu_layers=43,
)

In [ ]:
# https://ollama.com/library
# llm = Ollama(model='llama3.1:70b')

## Build the chain

In [ ]:
agent_colab = RetrievalQA.from_chain_type(llm=llm,verbose=True,chain_type="stuff",
                                          retriever=vector_store.as_retriever(search_kwargs={"k": 2}),
                                          chain_type_kwargs={"verbose": True})

## Run the query

In [ ]:
# The Company performs a detailed review of inventory each fiscal quarter that considers multiple factors including demand forecasts, product life cycle status, product development plans, current sales levels, and component cost trends.

query ="How often does the company review inventory, and what is considered in this inventory calculation?"

In [ ]:
output_colab = agent_colab.invoke({"query": query})



> Entering new RetrievalQA chain...


> Entering new StuffDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

have to adjust its allowance for doubtful accounts, which would affect earnings in the period the adjustments were made. Inventory Valuation and Inventory Purchase Commitments The Company must order components for its products and build inventory in advance of product shipments. The Company records a write-down for inventories of components and products, including third-party products held for resale, which have become obsolete or are in excess of anticipated demand or net realizable value. The Company performs a detailed review of inventory each fiscal quarter that considers multiple factors including demand forecasts, product life cycle status, product development plans, current sa

In [ ]:
print(output_colab)

{'query': 'How often does the company review inventory, and what is considered in this inventory calculation?', 'result': ' The company reviews inventory quarterly. It considers multiple factors such as demand forecasts, product life cycle status, product development plans, current sales levels, and component cost trends.'}
